# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a FAIR² Croissant dataset using the `mlcroissant` library. It specifically demonstrates exploration of mass spectrometry data from human extracellular vesicles.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Get the metadata object
metadata = dataset.metadata
print(f"Loaded dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets and fields (by their `@id`), enabling reference for later data extraction and processing.

In [ ]:
# List all record sets and their IDs
print("Available record sets (@id):\n")
for record_set in metadata.record_sets:
    print(f"  - {record_set.id}: {record_set.name}")

# For demonstration, print fields and their IDs for each record set
for record_set in metadata.record_sets:
    print(f"\nFields in {record_set.id} ({record_set.name}):")
    for field in record_set.fields:
        print(f"    - {field.id}: {field.name} (dataType: {field.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All operations here reference entities by their `@id` as shown in the overview.

In [ ]:
# Collect all record set @id's for easy access
record_sets_ids = [rs.id for rs in metadata.record_sets]

dataframes = {}
for rs_id in record_sets_ids:
    print(f"Loading records from RecordSet @id: {rs_id}")
    try:
        recs = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(recs)
        dataframes[rs_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Shape: {df.shape}\n")
    except Exception as e:
        print(f"  Could not load records for {rs_id}: {e}\n")

# For demonstration, inspect the columns of the primary data table (select record set 0)
main_record_set_id = record_sets_ids[0]
print(f"Primary record set columns: {dataframes[main_record_set_id].columns.tolist()}")
# Show first rows
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping. Use field `@id` references throughout.

In [ ]:
# Pick out a numeric field for analysis from the fields overview
# For this example, let's find a numeric field in the main record set
main_fields = [f for f in metadata.record_sets[0].fields]
for field in main_fields:
    if field.data_type in ['schema:Float', 'schema:Integer', 'schema:Number']:
        numeric_field_id = field.id
        print(f"Using numeric field for EDA: {numeric_field_id} ({field.name})")
        break

df = dataframes[main_record_set_id]
# Check if numeric_field_id exists in loaded columns
if numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean()  # Use mean as demonstration
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):\n")
    print(filtered_df.head())

    # Normalize the numeric column
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:\n")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to select a categorical field for grouping
    group_field_id = None
    for field in main_fields:
        if field.data_type in ['schema:Text', 'schema:String'] and field.id != numeric_field_id and field.id in df.columns:
            group_field_id = field.id
            print(f"Will demonstrate grouping by: {group_field_id} ({field.name})")
            break

    # Group data by chosen field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"No numeric field found in main record set for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of numeric field after filtering
if numeric_field_id in filtered_df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping was performed, plot bar chart
    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use `mlcroissant` to discover, access, and analyze mass spectrometry data described by a Croissant FAIR² schema. This workflow enables metadata-driven access to well-described scientific datasets, facilitating reproducible exploration and EDA using only the Croissant schema URL and unique `@id` references.

Next steps could include deeper statistical modeling, feature engineering, or integration with downstream ML workflows.